In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, mean_absolute_error
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.optimizers import Adam


In [2]:
df = pd.read_csv("prep_data.csv")
df = df.drop(columns=['followup1_date', 'followup2_date', 'surgery_date'])
X = df.drop(columns=[
    'followup2_vision_left', 'followup2_vision_right',
    'followup2_refraction_sph_left', 'followup2_refraction_cyl_left',
    'followup2_refraction_sph_right', 'followup2_refraction_cyl_right',
    'followup2_keratometry_left', 'followup2_keratometry_right',
    'followup1_complication', 'followup2_complication'
])
y_reg = df[[
    'followup2_vision_left', 'followup2_vision_right',
    'followup2_refraction_sph_left', 'followup2_refraction_cyl_left',
    'followup2_refraction_sph_right', 'followup2_refraction_cyl_right',
    'followup2_keratometry_left', 'followup2_keratometry_right'
]]
y_class1 = df['followup1_complication']
y_class2 = df['followup2_complication']
X_train, X_test, y_reg_train, y_reg_test, y_class1_train, y_class1_test, y_class2_train, y_class2_test = train_test_split(
    X, y_reg, y_class1, y_class2, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
input_reg = Input(shape=(X_train.shape[1],))
x = Dense(128, activation='relu')(input_reg)
x = Dropout(0.4)(x)
x = Dense(64, activation='relu')(x)
output_reg = Dense(8, name='regression')(x)
reg_model = Model(inputs=input_reg, outputs=output_reg)
reg_model.compile(optimizer=Adam(0.001), loss='mse', metrics=['mae'])
reg_model.summary()
reg_history = reg_model.fit(
    X_train,
    y_reg_train,
    validation_split=0.1,
    epochs=45,
    batch_size=32,
    verbose=1
)
reg_preds = reg_model.predict(X_test)
print("\nMean Absolute Error on Regression Test Set:", mean_absolute_error(y_reg_test, reg_preds))



Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)             │ (None, 37)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │           4,864 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ regression (Dense)                   │ (None, 8)                   │             520 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 13,640 (53.28 KB)

 Trainable params: 13,640 (53.28 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/45
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step - loss: 486.8576 - mae: 11.7394 - val_loss: 466.4830 - val_mae: 11.4547
Epoch 2/45
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 465.5870 - mae: 11.4783 - val_loss: 444.6080 - val_mae: 11.1925
Epoch 3/45
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 443.0816 - mae: 11.2100 - val_loss: 418.2227 - val_mae: 10.8782
Epoch 4/45
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 413.9926 - mae: 10.8730 - val_loss: 385.3272 - val_mae: 10.4558
Epoch 5/45
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 377.0053 - mae: 10.3932 - val_loss: 345.1705 - val_mae: 9.8746
Epoch 6/45
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 334.4919 - mae: 9.8117 - val_loss: 297.9078 - val_mae: 9.1906
Epoch 7/45
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 284.8950 - mae: 9.0629 - val_loss: 245.1951 - val_mae: 8.4013
Epoch 8/45
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 225.3533 - mae: 8.1274 - val_loss: 189.4579 - val_mae: 7.4219
Epoch 9/45
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/ste

In [3]:
input_class = Input(shape=(X_train.shape[1],))
x = Dense(128, activation='relu')(input_class)
x = Dropout(0.45)(x)
x = Dense(64, activation='relu')(x)
output_class1 = Dense(len(np.unique(y_class1)), activation='softmax', name='class1')(x)
output_class2 = Dense(len(np.unique(y_class2)), activation='softmax', name='class2')(x)
class_model = Model(inputs=input_class, outputs=[output_class1, output_class2])
class_model.compile(
    optimizer='adam',
    loss={
        'class1': 'sparse_categorical_crossentropy',
        'class2': 'sparse_categorical_crossentropy'
    },
    metrics={
        'class1': 'accuracy',
        'class2': 'accuracy'
    }
)

class_model.summary()
class_history = class_model.fit(
    X_train,
    [y_class1_train, y_class2_train],
    validation_split=0.1,
    epochs=35,
    batch_size=32,
    verbose=1
)
class_preds = class_model.predict(X_test)
pred_class1 = np.argmax(class_preds[0], axis=1)
pred_class2 = np.argmax(class_preds[1], axis=1)
print("\nClassification Report for Followup1 Complications:")
print(classification_report(y_class1_test, pred_class1))
print("\nClassification Report for Followup2 Complications:")
print(classification_report(y_class2_test, pred_class2))


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)    │ (None, 37)                │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_2 (Dense)               │ (None, 128)               │           4,864 │ input_layer_1[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_1 (Dropout)           │ (None, 128)               │               0 │ dense_2[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_3 (Dense)               │ (None, 64)                │           8,256 │ dropout_1[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ class1 (Dense)                │ (None, 6)                 │             390 │ dense_3[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ class2 (Dense)                │ (None, 6)                 │             390 │ dense_3[0][0]              │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 13,900 (54.30 KB)

 Trainable params: 13,900 (54.30 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/35
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - class1_accuracy: 0.1037 - class1_loss: 2.0456 - class2_accuracy: 0.0800 - class2_loss: 2.0998 - loss: 4.1479 - val_class1_accuracy: 0.4286 - val_class1_loss: 1.5648 - val_class2_accuracy: 0.5000 - val_class2_loss: 1.6879 - val_loss: 3.2527
Epoch 2/35
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - class1_accuracy: 0.4434 - class1_loss: 1.5443 - class2_accuracy: 0.5080 - class2_loss: 1.3855 - loss: 2.9304 - val_class1_accuracy: 0.7500 - val_class1_loss: 1.1548 - val_class2_accuracy: 0.6429 - val_class2_loss: 1.4778 - val_loss: 2.6326
Epoch 3/35
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - class1_accuracy: 0.7021 - class1_loss: 1.0999 - class2_accuracy: 0.7871 - class2_loss: 1.0076 - loss: 2.1077 - val_class1_accuracy: 0.8571 - val_class1_loss: 0.8491 - val_class2_accuracy: 0.6429 - val_class2_loss: 1.4766 - val_loss: 2.3256
Epoch 4/35
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - class1_accuracy: 0.7851 - class1_loss: 0.9958 - class2_accuracy: 0.8161 - clas

C:\Users\Lenovo_lappy\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Lenovo_lappy\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Lenovo_lappy\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{m

In [6]:
reg_model.save("regression_model.h5")
class_model.save("classification_model.h5")